# validate_epss
Bronze EPSS(`CSV.gz`, 전량) → Valid / Reject. `active_cve_set` 필터링은 다음 단계(`build_silver_epss`)에서 한다 — 이 단계는 오직 CVE/score/date 형식 검증만 한다.
- **Fatal**: `cve`/`cve_id` 없음, `epss`/`percentile`이 숫자로 변환 불가.
- **Warn**: 값이 있지만 0~1 범위를 벗어남(`EPSS_OUT_OF_RANGE`, `PERCENTILE_OUT_OF_RANGE`) — Reject하지 않고 플래그만 남겨 `build_silver_epss`가 dq_flags로 이어받는다.
- `snapshot_date`는 CSV 헤더 주석(`#score_date:...`)에서, 없으면 파일 경로의 날짜 패턴에서 추출한다.

In [ ]:
%run 00_silver_common

In [ ]:
import re

epss_dir = f"{BRONZE_EPSS_PATH}/{snapshot_date}"

# 헤더 주석(#model_version:..., #score_date:...)은 CSV 데이터 행이 아니라 파일 맨 위 1줄에 별도로 온다.
header_line = ""
try:
    header_rdd = spark.sparkContext.textFile(f"{epss_dir}/*.csv.gz")
    first_lines = header_rdd.filter(lambda l: l.startswith("#")).take(5)
    header_line = " ".join(first_lines)
except Exception:
    header_line = ""

def parse_header_field(text: str, key: str):
    m = re.search(rf"{key}\s*:\s*([^,\s]+)", text)
    return m.group(1) if m else None

header_score_date = parse_header_field(header_line, "score_date")
header_model_version = parse_header_field(header_line, "model_version")

path_date_match = re.search(r"(\d{4}-\d{2}-\d{2})", epss_dir)
resolved_snapshot_date = header_score_date or (path_date_match.group(1) if path_date_match else snapshot_date)


In [ ]:
raw_df = (
    spark.read
    .option("header", "true")
    .option("comment", "#")
    .csv(f"{epss_dir}/*.csv.gz")
)

cve_col = coalesce_existing(raw_df, ["cve", "cve_id"])
score_col = coalesce_existing(raw_df, ["epss"]).cast(DoubleType())
percentile_col = coalesce_existing(raw_df, ["percentile"]).cast(DoubleType())

raw_df = (
    raw_df
    .withColumn("_cve_id", cve_col)
    .withColumn("_epss_score", score_col)
    .withColumn("_epss_percentile", percentile_col)
)

fatal_cond = F.col("_cve_id").isNull() | F.col("_epss_score").isNull() | F.col("_epss_percentile").isNull()

warn_flags = flags_array(
    F.when(
        F.col("_epss_score").isNotNull() & ((F.col("_epss_score") < 0) | (F.col("_epss_score") > 1)),
        F.lit("EPSS_OUT_OF_RANGE"),
    ),
    F.when(
        F.col("_epss_percentile").isNotNull() & ((F.col("_epss_percentile") < 0) | (F.col("_epss_percentile") > 1)),
        F.lit("PERCENTILE_OUT_OF_RANGE"),
    ),
)
raw_df = raw_df.withColumn("_validation_flags", warn_flags)


In [ ]:
reject_df = (
    raw_df.filter(fatal_cond)
    .withColumn("reject_reason", F.lit("MISSING_OR_INVALID_SCORE"))
    .withColumn("raw_payload", F.to_json(F.struct(*[c for c in raw_df.columns if not c.startswith("_")])))
    .withColumn("source", F.lit("epss_official"))
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("ingested_at", F.lit(INGESTED_AT))
    .select("raw_payload", "reject_reason", "source", "batch_id", "ingested_at")
)

valid_df = (
    raw_df.filter(~fatal_cond)
    .select(
        F.col("_cve_id").alias("cve_id"),
        F.col("_epss_score").alias("epss_score"),
        F.col("_epss_percentile").alias("epss_percentile"),
        F.col("_validation_flags").alias("validation_flags"),
    )
    .withColumn("snapshot_date", F.lit(resolved_snapshot_date))
    .withColumn("model_version", F.lit(header_model_version))
    .withColumn("source", F.lit("epss_official"))
    .withColumn("batch_id", F.lit(batch_id))
    .withColumn("ingested_at", F.lit(INGESTED_AT))
)

reject_count = reject_df.count()
valid_count = valid_df.count()

if reject_count > 0:
    write_append(reject_df, TBL["reject_epss"])
write_overwrite(valid_df, TBL["stg_epss_valid"])

log_batch_run("validate_epss", "SUCCESS", record_count=valid_count)
log_dq_metric("validate_epss", "reject_rate", reject_count / max(valid_count + reject_count, 1), passed=True,
              detail=f"valid={valid_count}, reject={reject_count}, snapshot_date={resolved_snapshot_date}")
print(f"[validate_epss] valid={valid_count} reject={reject_count} snapshot_date={resolved_snapshot_date} batch_id={batch_id}")
